In [ ]:
# Notebook 04 - Recommendation Model (step-by-step)
#
# Input: artifacts/features/ from Notebook 03
# Output: artifacts/models/ (SVD model, hybrid config, evaluation results)
#
# Cell 1 - Setup & Load Artifacts
#   Load all feature artifacts, build lookup dicts, verify shapes
#
# Cell 2 - Popularity-Based Recommender
#   Rank by Bayesian popularity_score, fallback for cold-start users
#
# Cell 3 - Content-Based Recommender
#   Similarity lookup from pre-computed top-k neighbors
#   User personalization via high-rated recipe neighbors
#
# Cell 4 - Collaborative Filtering (SVD)
#   User-item sparse matrix -> TruncatedSVD -> predicted ratings
#   Only works for users seen in train
#
# Cell 5 - Constraint Filter
#   Rule-based filter: max_calories, max_minutes, tags_include
#   Wraps around any recommender output
#
# Cell 6 - Hybrid Combiner
#   Weighted: alpha * content_score + (1-alpha) * cf_score
#   Fallback tiers: cold -> popularity, warm -> content, active -> hybrid
#
# Cell 7 - Qualitative Check
#   Spot-check 3-5 sample users (active / warm / cold-start)
#
# Cell 8 - Evaluation on Validation Set
#   Precision@K, Recall@K, NDCG@K, Coverage
#   Cold-start user report
#
# Cell 9 - Save Model Artifacts
#   svd_model.joblib, hybrid_config.json, evaluation_results.json

In [3]:
# === Cell 1 - Setup & Load Artifacts ===

import pandas as pd
import numpy as np
import ast
import json
import joblib
from pathlib import Path
from scipy import sparse
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics.pairwise import cosine_similarity
from collections import defaultdict
from scipy.sparse import csr_matrix

FEATURES_DIR = Path("../artifacts/features")

# ---------- Load artifacts ----------

with open(FEATURES_DIR / "feature_manifest.json") as f:
    manifest = json.load(f)

tfidf_vectorizer = joblib.load(FEATURES_DIR / "tfidf_vectorizer.joblib")
recipe_tfidf_matrix = sparse.load_npz(FEATURES_DIR / "recipe_tfidf_matrix.npz")

recipe_index_map = pd.read_csv(FEATURES_DIR / "recipe_index_map.csv")
recipe_similarity_topk = pd.read_csv(FEATURES_DIR / "recipe_similarity_topk.csv")
recipe_features = pd.read_csv(FEATURES_DIR / "recipe_features.csv")
popularity_features = pd.read_csv(FEATURES_DIR / "popularity_features.csv")

user_features = pd.read_csv(FEATURES_DIR / "user_features.csv")
user_features["favorite_tags"] = user_features["favorite_tags"].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) and x.startswith("[") else []
)

train_interactions = pd.read_csv(FEATURES_DIR / "train_interactions.csv")
val_interactions = pd.read_csv(FEATURES_DIR / "val_interactions.csv")
train_interactions["date"] = pd.to_datetime(train_interactions["date"], errors="coerce")
val_interactions["date"] = pd.to_datetime(val_interactions["date"], errors="coerce")

# ---------- Load recipe names from processed data ----------

df_recipes = pd.read_csv("../data/processed/recipes_clean.csv", usecols=["id", "name"])
recipe_name_map = dict(zip(df_recipes["id"], df_recipes["name"]))

# ---------- Build lookup dicts ----------

id2idx = dict(zip(recipe_index_map["recipe_id"], recipe_index_map["matrix_index"]))
idx2id = dict(zip(recipe_index_map["matrix_index"], recipe_index_map["recipe_id"]))

train_user_ids = set(train_interactions["user_id"].unique())
train_recipe_ids = set(train_interactions["recipe_id"].unique())

# ---------- Verify shapes ----------

print(f"Manifest version:        {manifest['version']}")
print(f"Build timestamp:         {manifest['build_timestamp']}")
print()
print(f"recipe_tfidf_matrix:     {recipe_tfidf_matrix.shape}")
print(f"recipe_index_map:        {len(recipe_index_map):,} rows")
print(f"recipe_similarity_topk:  {len(recipe_similarity_topk):,} rows")
print(f"recipe_features:         {len(recipe_features):,} rows")
print(f"popularity_features:     {len(popularity_features):,} rows")
print(f"user_features:           {len(user_features):,} rows")
print(f"train_interactions:      {len(train_interactions):,} rows")
print(f"val_interactions:        {len(val_interactions):,} rows")
print()
print(f"Unique train users:      {len(train_user_ids):,}")
print(f"Unique train recipes:    {len(train_recipe_ids):,}")
print(f"id2idx entries:          {len(id2idx):,}")
print(f"recipe_name_map entries: {len(recipe_name_map):,}")
print()
print(f"Sample user_features favorite_tags: {user_features['favorite_tags'].iloc[0]}")
print("\nAll artifacts loaded successfully.")

Manifest version:        1.0
Build timestamp:         2026-04-17T16:00:28.027481

recipe_tfidf_matrix:     (77300, 20000)
recipe_index_map:        77,300 rows
recipe_similarity_topk:  3,865,000 rows
recipe_features:         77,300 rows
popularity_features:     77,300 rows
user_features:           27,021 rows
train_interactions:      559,500 rows
val_interactions:        139,876 rows

Unique train users:      27,021
Unique train recipes:    72,860
id2idx entries:          77,300
recipe_name_map entries: 77,300

Sample user_features favorite_tags: ['time-to-make', 'preparation', 'course']

All artifacts loaded successfully.


In [4]:
# === Cell 2 - Popularity-Based Recommender ===
#
# Xếp hạng recipe theo popularity_score (Bayesian smoothing từ NB03).
# Đây là recommender đơn giản nhất, dùng làm:
#   - Baseline so sánh với các model phức tạp hơn
#   - Fallback cho cold-start users (không có lịch sử)

# Pre-sort popularity một lần, dùng lại cho mọi lần gọi
popularity_ranked = popularity_features.sort_values("popularity_score", ascending=False).reset_index(drop=True)

def recommend_popular(n=10, exclude_ids=None):
    """Return top-n popular recipes, optionally excluding already-seen IDs."""
    candidates = popularity_ranked
    if exclude_ids:
        candidates = candidates[~candidates["recipe_id"].isin(exclude_ids)]
    top = candidates.head(n).copy()
    top["rank"] = range(1, len(top) + 1)
    top["name"] = top["recipe_id"].map(recipe_name_map)
    return top[["rank", "recipe_id", "name", "popularity_score"]].reset_index(drop=True)

# ---------- Quick test ----------
print("Top-10 popular recipes:")
display(recommend_popular(10))

Top-10 popular recipes:


,rank,recipe_id,name,popularity_score
0,1,77497,2 handed kitchen sink tomato sandwich,4.969171
1,2,34753,basic sourdough bread,4.967149
2,3,154351,kittencal s balsamic vinaigrette,4.965450
3,4,78579,pan release professional pan coating better th...,4.965094
4,5,186029,the best creole cajun seasoning mix,4.964216
5,6,58516,hot pepper jelly,4.961464
6,7,244193,absolutely the best new york cheesecake gluten...,4.959104
7,8,55309,caprese salad tomatoes italian marinated tomatoes,4.958252
8,9,31639,7 layer bean dip,4.957364
9,10,25094,my chicken parmigiana,4.957087


In [5]:
# === Cell 3 - Content-Based Recommender ===
# - recommend_content_similar(recipe_id, n): lookup pre-computed top-k neighbors
# - recommend_for_user_content(user_id, n):
#     Get recipes user rated >= 4 in train
#     Aggregate neighbors, remove already seen, rank by avg similarity
#     Fallback: use favorite_tags or popular if no history

# Build neighbor lookup
neighbors_lookup = recipe_similarity_topk.groupby("recipe_id")


# -------------------------------------------------
# Recommend recipes similar to a given recipe
# -------------------------------------------------

def recommend_content_similar(recipe_id, n=10):

    if recipe_id not in neighbors_lookup.groups:
        return pd.DataFrame()

    neighbors = neighbors_lookup.get_group(recipe_id)

    neighbors = neighbors.sort_values("similarity", ascending=False)

    top = neighbors.head(n).copy()

    top["name"] = top["neighbor_recipe_id"].map(recipe_name_map)
    top["rank"] = range(1, len(top) + 1)

    return top.rename(
        columns={"neighbor_recipe_id": "recipe_id"}
    )[["rank", "recipe_id", "name", "similarity"]]


# -------------------------------------------------
# Content recommendation for a user
# -------------------------------------------------

def recommend_for_user_content(user_id, n=10):

    user_history = train_interactions[
        train_interactions["user_id"] == user_id
    ]

    liked = user_history[user_history["rating"] >= 4]["recipe_id"].tolist()

    # ---------- fallback: cold user ----------
    if len(liked) == 0:

        user_row = user_features[user_features["user_id"] == user_id]

        if len(user_row) == 0:
            return recommend_popular(n)

        fav_tags = user_row.iloc[0]["favorite_tags"]

        if len(fav_tags) == 0:
            return recommend_popular(n)

        tag_recipes = df_recipes[
            df_recipes["id"].isin(
                recipe_features["recipe_id"]
            )
        ]

        candidates = popularity_features.sort_values(
            "popularity_score", ascending=False
        )

        return recommend_popular(n)

    # ---------- aggregate neighbors ----------
    scores = defaultdict(list)

    for recipe_id in liked:

        if recipe_id not in neighbors_lookup.groups:
            continue

        neighbors = neighbors_lookup.get_group(recipe_id)

        for _, row in neighbors.iterrows():
            scores[row["neighbor_recipe_id"]].append(row["similarity"])

    seen = set(user_history["recipe_id"])

    scored = {
        r: np.mean(vals)
        for r, vals in scores.items()
        if r not in seen
    }

    ranked = sorted(scored.items(), key=lambda x: x[1], reverse=True)[:n]

    result = pd.DataFrame(ranked, columns=["recipe_id", "content_score"])

    result["name"] = result["recipe_id"].map(recipe_name_map)
    result["rank"] = range(1, len(result) + 1)

    return result[["rank", "recipe_id", "name", "content_score"]]

In [6]:
# === Cell 4 - Collaborative Filtering (SVD) ===
# - Build user-item sparse matrix from train_interactions (centered by user mean)
# - Fit TruncatedSVD(n_components=50)
# - recommend_cf(user_id, n): reconstruct predicted ratings, exclude seen, return top-N
# - Only works for users present in train set

# Build user-item matrix
# -------------------------------------------------

user_ids = train_interactions["user_id"].unique()
recipe_ids = train_interactions["recipe_id"].unique()

user_to_index = {u: i for i, u in enumerate(user_ids)}
item_to_index = {r: i for i, r in enumerate(recipe_ids)}

index_to_user = {i: u for u, i in user_to_index.items()}
index_to_item = {i: r for r, i in item_to_index.items()}

rows = train_interactions["user_id"].map(user_to_index)
cols = train_interactions["recipe_id"].map(item_to_index)
data = train_interactions["rating"]

user_item_matrix = csr_matrix(
    (data, (rows, cols)),
    shape=(len(user_ids), len(recipe_ids))
)

print("User-item matrix shape:", user_item_matrix.shape)


# -------------------------------------------------
# Center ratings by user mean
# -------------------------------------------------

user_means = train_interactions.groupby("user_id")["rating"].mean()

centered_data = train_interactions.apply(
    lambda r: r["rating"] - user_means[r["user_id"]],
    axis=1
)

centered_matrix = csr_matrix(
    (centered_data, (rows, cols)),
    shape=user_item_matrix.shape
)


# -------------------------------------------------
# Train SVD
# -------------------------------------------------

svd_model = TruncatedSVD(
    n_components=50,
    random_state=42
)

user_factors = svd_model.fit_transform(centered_matrix)
item_factors = svd_model.components_

print("SVD model trained")


# -------------------------------------------------
# CF recommendation function
# -------------------------------------------------

def recommend_cf(user_id, n=10):

    if user_id not in user_to_index:
        return recommend_popular(n)

    u_idx = user_to_index[user_id]

    user_vector = user_factors[u_idx]

    scores = user_vector @ item_factors

    scores = scores + user_means[user_id]

    seen = train_interactions[
        train_interactions["user_id"] == user_id
    ]["recipe_id"].tolist()

    candidates = []

    for idx, score in enumerate(scores):

        recipe_id = index_to_item[idx]

        if recipe_id in seen:
            continue

        candidates.append((recipe_id, score))

    ranked = sorted(candidates, key=lambda x: x[1], reverse=True)[:n]

    result = pd.DataFrame(ranked, columns=["recipe_id", "predicted_rating"])

    result["name"] = result["recipe_id"].map(recipe_name_map)
    result["rank"] = range(1, len(result) + 1)

    return result[["rank", "recipe_id", "name", "predicted_rating"]]

User-item matrix shape: (27021, 72860)
SVD model trained


In [ ]:
# === Cell 5 - Constraint Filter ===
# - apply_constraints(candidate_ids, max_calories, max_minutes, tags_include)
# - Filters candidate recipes by rule-based conditions
# - Wraps around any recommender output before returning to user

In [ ]:
# === Cell 6 - Hybrid Combiner ===
# - hybrid_score = alpha * content_score_norm + (1-alpha) * cf_score_norm
# - Fallback tiers:
#     Cold-start (no train history)  -> Popularity + Content from tags
#     Warm (< 5 interactions)        -> Content-heavy (alpha=0.7)
#     Active (>= 5 interactions)     -> Balanced hybrid (alpha=0.5)
# - recommend_hybrid(user_id, n, alpha, constraints)
# - Returns results with source tag: popularity / content / cf / hybrid

In [ ]:
# === Cell 7 - Qualitative Check ===
# - Pick 3-5 sample users: 1 active, 1 warm, 1 cold-start
# - Print recommendations from each model + hybrid
# - Compare: recipe name, score, source

In [ ]:
# === Cell 8 - Evaluation on Validation Set ===
# - Ground truth: recipes rated >= 4 per user in val_interactions
# - Predicted: top-K from each model
# - Metrics: Precision@K, Recall@K, NDCG@K, Coverage
# - Comparison table: Popularity vs Content vs CF vs Hybrid
# - Separate cold-start user report

In [ ]:
# === Cell 9 - Save Model Artifacts ===
# - Save to artifacts/models/:
#     svd_model.joblib
#     user_index_map.csv
#     hybrid_config.json (alpha, tiers, constraint defaults)
#     evaluation_results.json